Image CNN with transfer learning from MobileNETV2 and FCNN head for classification:

Structure/Workflow:

Image ->
MobileNetV2 (pretrained) ->
Global pooled features (1280-d) ->
Projection layer (optional, recommended) ->
Classifier head (FCNN)


Expected Folder Strucure(expand to read):

dataset_root/
├── train/
│   ├── Blue/
│   │   ├── image_001.jpg
│   │   ├── image_002.jpg
│   │   └── ...
│   ├── Green/
│   │   └── ...
│   ├── Black/
│   │   └── ...
│   └── Other/
│       └── ...
├── val/
│   ├── Blue/
│   ├── Green/
│   ├── Black/
│   └── Other/
└── test/
    ├── Blue/
    ├── Green/
    ├── Black/
    └── Other/


Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import os

Device Setup

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda:0


Paths/Parameters/Settings definitions

In [ ]:
BATCH_SIZE = 8
NUM_WORKERS = 2
NUM_CLASSES = 4
EPOCHS = 30
UNFREEZE_EPOCH = int(0.8*EPOCHS)
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
SAVE_MODEL_PATH = "/home/le.song/Assignment2/best_image_model.pth"
MISCLASS_FILE = "/home/le.song/Assignment2/misclassified.txt"
LOSS_PLOT_PATH = "/home/le.song/Assignment2/loss_curve.png"
TRAIN_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Train"
VAL_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Val"
TEST_PATH = "/work/TALC/ensf617_2026w/garbage_data/CVPR_2024_dataset_Test"

Data Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), #Optional, will monitor effects.
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])  #Stats need to align with that of resnet, no stat calculations needed on input data.

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  #Testing data MUST NOT be pre-processed. No rotations/flips/jitter applied.
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


 Datasets and Data-Loaders

In [ ]:
import shutil # Added for removing directories

def remove_ipynb_checkpoints(root_dir):
    if not os.path.exists(root_dir):
        print(f"Warning: Root directory {root_dir} does not exist. Skipping cleanup.")
        return
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if '.ipynb_checkpoints' in dirnames:
            checkpoints_path = os.path.join(dirpath, '.ipynb_checkpoints')
            print(f"Removing .ipynb_checkpoints directory: {checkpoints_path}")
            shutil.rmtree(checkpoints_path)
            # Remove from dirnames so os.walk doesn't try to enter it after deletion
            dirnames.remove('.ipynb_checkpoints')

# Applied as cleanup before creating ImageFolder datasets
remove_ipynb_checkpoints(TRAIN_PATH)
remove_ipynb_checkpoints(VAL_PATH)
remove_ipynb_checkpoints(TEST_PATH)

train_dataset = ImageFolder(root=TRAIN_PATH, transform=train_transform)
val_dataset = ImageFolder(root=VAL_PATH, transform=test_transform)
test_dataset = ImageFolder(root=TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#Verifying sizes/dimension
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 200
Val samples: 205
Test samples: 161


MobileNetV2 Image Encoder Definition

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ImageEncoder(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True, proj_dim=256):
        super().__init__()

        # Load pretrained ResNet50
        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pretrained else None
        )

        # Remove final classification layer
        self.feature_dim = self.backbone.fc.in_features  # 2048 for ResNet50
        self.backbone.fc = nn.Identity()

        # Optional freezing
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Projection layer (aligns with text embedding size)
        self.projection = nn.Sequential(
            nn.Linear(self.feature_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU()
        )

    def forward(self, x):
        features = self.backbone(x)      # [B, 2048]
        projected = self.projection(features)  # [B, proj_dim]
        return projected

Image Classifier Definition

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, encoder, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
          nn.Linear(256, 64),
          nn.ReLU(),
          nn.Dropout(0.2),
          nn.Linear(64, num_classes)
        )


    def forward(self, x):
        embeddings = self.encoder(x)
        logits = self.classifier(embeddings)
        return logits

Initialize

In [ ]:
encoder = ImageEncoder(pretrained=True, freeze_backbone=True, proj_dim=256)
model = ImageClassifier(encoder).to(device)

Training Loop Definition: Few words on this part. Initial design was to manually freeze/unfreeze pretrained weights as demonstrated in class, however, due to considerations given to potential need to submit to TALC, such design became infeasible because the possibility of having to submit MULTIPLE jobs and having ANOTHER submission come in between our submissions and I'm unsure how it will affect the model's performance, or can the save point be even accessed correctly if a different request was processed in between. Decision: make the training/validation loop automatically time freeze/unfreeze and process the entire process in 1 submission. In between freeze/unfreeze the optimizer learning rate as well as # of transfer learning epochs will also be tuned to avoid overfit.

In [ ]:
def train_model(model, train_loader, val_loader, device,
                epochs=EPOCHS,               # initial # of epochs used for model initialization
                unfreeze_epoch=UNFREEZE_EPOCH,  # epoch after which the backbone is unfrozen
                initial_lr=INITIAL_LR,       # learning rate before unfreeze
                fine_tune_lr=FINE_TUNE_LR,   # learning rate after unfreeze
                save_path=SAVE_MODEL_PATH):  # file path to save best val-loss model, Saving to FILE may be required, the model will be used later in multi-modal training once combined with text portion.

    # ---- Loss ----
    criterion = nn.CrossEntropyLoss()

    # ---- Initial optimizer: AdamW ----
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=initial_lr,
        weight_decay=1e-4
    )

    train_losses, val_losses = [], []
    best_val_loss = float("inf")

    for epoch in range(epochs):

        # This is how the automatic unfreeze works, once it reaches the defined epoch, backbone unfreezes and AdamW's lr is changed to fine_tune_lr.
        if epoch == unfreeze_epoch:
            print("Unfreezing backbone...")
            for param in model.encoder.backbone.parameters():
                param.requires_grad = True # Unfreeze happens

            # Updating lr in AdamW
            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=fine_tune_lr,
                weight_decay=1e-4
            )

        #Training
        model.train()
        running_train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad() #resets gradient between batches
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step() #Steps only after back-propagation
            running_train_loss += loss.item()
        epoch_train_loss = running_train_loss / len(train_loader) #training loss
        train_losses.append(epoch_train_loss)

        #Validation
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
        epoch_val_loss = running_val_loss / len(val_loader) #validation loss
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1}/{epochs} " #Prints losses
              f"Train Loss: {epoch_train_loss:.4f} "
              f"Val Loss: {epoch_val_loss:.4f}")

        # Saving model with smallest val_loss
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), save_path)
            print("Saved best model.")

    return train_losses, val_losses


Full-filling requirements to track misclassified images(Unimodal)

In [ ]:
def evaluate_and_log_errors(model, test_loader, device,
                            save_model_path=SAVE_MODEL_PATH,
                            error_log_file=MISCLASS_FILE):

    model.load_state_dict(torch.load(save_model_path))
    model.eval()

    incorrect_samples = []
    correct, total = 0, 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0) #Automatic resizing
            correct += (preds == labels).sum().item()

            # Log misclassified images
            for i in range(len(labels)):
                if preds[i] != labels[i]:
                    dataset_index = batch_idx * test_loader.batch_size + i
                    path, _ = test_loader.dataset.samples[dataset_index]
                    incorrect_samples.append(
                        f"{path} | Pred: {preds[i].item()} | True: {labels[i].item()}"
                    )

    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")

    with open(error_log_file, "w") as f:
        for line in incorrect_samples:
            f.write(line + "\n")
    print(f"Misclassified samples written to {error_log_file}")
    return accuracy

Loss Curve

In [ ]:
def plot_losses(train_losses, val_losses, save_path=LOSS_PLOT_PATH):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Training and Validation Loss")
    plt.savefig(save_path)
    plt.close()
    print(f"Loss curve saved to {save_path}")

Running model

In [ ]:
train_losses, val_losses = train_model(model, train_loader, val_loader, device)
plot_losses(train_losses, val_losses)
accuracy = evaluate_and_log_errors(model, test_loader, device)
imagemodel = model

Epoch 1/30 Train Loss: 1.2863 Val Loss: 1.2244
Saved best model.
Epoch 2/30 Train Loss: 0.9867 Val Loss: 1.0479
Saved best model.
Epoch 3/30 Train Loss: 0.7630 Val Loss: 1.0555
Epoch 4/30 Train Loss: 0.6719 Val Loss: 1.0646
Epoch 5/30 Train Loss: 0.5884 Val Loss: 1.1415
Epoch 6/30 Train Loss: 0.5492 Val Loss: 1.1402
Epoch 7/30 Train Loss: 0.4881 Val Loss: 1.2106
Epoch 8/30 Train Loss: 0.5145 Val Loss: 1.2493
Epoch 9/30 Train Loss: 0.3640 Val Loss: 1.3435
Epoch 10/30 Train Loss: 0.3563 Val Loss: 1.3346
Epoch 11/30 Train Loss: 0.2795 Val Loss: 1.3579
Epoch 12/30 Train Loss: 0.3252 Val Loss: 1.3922
Epoch 13/30 Train Loss: 0.3872 Val Loss: 1.3999
Epoch 14/30 Train Loss: 0.2876 Val Loss: 1.2275
Epoch 15/30 Train Loss: 0.3588 Val Loss: 1.3622
Epoch 16/30 Train Loss: 0.2847 Val Loss: 1.3425
Epoch 17/30 Train Loss: 0.3078 Val Loss: 1.5503
Epoch 18/30 Train Loss: 0.3032 Val Loss: 1.4024
Epoch 19/30 Train Loss: 0.1978 Val Loss: 1.4638
Epoch 20/30 Train Loss: 0.2145 Val Loss: 1.5913
Epoch 21/30 T

Retrieve Image Embeddings for Fusion

In [ ]:
def get_image_embeddings_batchwise(dataset, model, batch_size=16):
    """
    Extract image embeddings in batches to avoid OOM errors.
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_embeddings = []

    with torch.no_grad():
        for batch in loader:
            images, _ = batch  # ignore labels here
            images = images.to(device)
            embeddings = model.encoder(images)
            all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0).numpy()